In [1]:
"""
Solutions to the end-of-chapter exercises (Géron, Hands-On ML, Ch.7 — Ensemble Learning)

Exercise 8: Train RandomForest / ExtraTrees / SVM on MNIST, combine into a
            voting ensemble (hard or soft), compare to individual classifiers.
Exercise 9: Build a stacking ensemble by hand (train a "blender" on the
            individual classifiers' validation predictions), compare to the
            voting classifier, then compare again using sklearn's
            StackingClassifier.
"""


'\nSolutions to the end-of-chapter exercises (Géron, Hands-On ML, Ch.7 — Ensemble Learning)\n\nExercise 8: Train RandomForest / ExtraTrees / SVM on MNIST, combine into a\n            voting ensemble (hard or soft), compare to individual classifiers.\nExercise 9: Build a stacking ensemble by hand (train a "blender" on the\n            individual classifiers\' validation predictions), compare to the\n            voting classifier, then compare again using sklearn\'s\n            StackingClassifier.\n'

In [2]:

import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier,
    VotingClassifier, StackingClassifier,
)
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [3]:
# 0. Load MNIST and split 50k/10k/10k (train/valid/test)
# ---------------------------------------------------------------------------
def load_mnist_splits(random_state=42):
    mnist = fetch_openml("mnist_784", version=1, as_frame=False)
    X, y = mnist["data"], mnist["target"].astype(np.uint8)

    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X, y, test_size=10000, random_state=random_state
    )
    X_train, X_valid, y_train, y_valid = train_test_split(
        X_train_full, y_train_full, test_size=10000, random_state=random_state
    )
    return X_train, y_train, X_valid, y_valid, X_test, y_test



In [4]:
# ---------------------------------------------------------------------------
# 8. Voting ensemble
# ---------------------------------------------------------------------------
def exercise_8(X_train, y_train, X_valid, y_valid, X_test, y_test,
               random_state=42):
    random_forest_clf = RandomForestClassifier(n_estimators=100, random_state=random_state)
    extra_trees_clf = ExtraTreesClassifier(n_estimators=100, random_state=random_state)
    # LinearSVC often needs many more iterations than the default (1000) to
    # actually converge on 784-dim pixel data — too few iterations silently
    # produces a near-untrained, near-random classifier rather than an error.
    svm_clf = LinearSVC(max_iter=2000, tol=1e-3, random_state=random_state)

    estimators = [random_forest_clf, extra_trees_clf, svm_clf]
    for estimator in estimators:
        print("Training", estimator.__class__.__name__)
        estimator.fit(X_train, y_train)

    individual_val_scores = [
        estimator.score(X_valid, y_valid) for estimator in estimators
    ]
    print("Ex8 - Individual validation accuracies:",
          dict(zip([e.__class__.__name__ for e in estimators], individual_val_scores)))
    # LinearSVC is typically the weak link here (linear boundary on raw
    # pixels), noticeably behind the two tree ensembles.

    named_estimators = [
        ("random_forest_clf", random_forest_clf),
        ("extra_trees_clf", extra_trees_clf),
        ("svm_clf", svm_clf),
    ]

    voting_clf_hard = VotingClassifier(named_estimators, voting="hard")
    voting_clf_hard.fit(X_train, y_train)
    hard_val_acc = voting_clf_hard.score(X_valid, y_valid)
    print("Ex8 - Hard voting validation accuracy:", hard_val_acc)

    # LinearSVC doesn't natively output probabilities, so soft voting isn't
    # directly available for it. Two options: drop it from the soft-voting
    # ensemble, or wrap it with probability estimates via CalibratedClassifierCV.
    # Simplest fix matching the book: just exclude it from the soft-voting set.
    voting_clf_soft = VotingClassifier(
        [("random_forest_clf", random_forest_clf),
         ("extra_trees_clf", extra_trees_clf)],
        voting="soft",
    )
    voting_clf_soft.fit(X_train, y_train)
    soft_val_acc = voting_clf_soft.score(X_valid, y_valid)
    print("Ex8 - Soft voting (RF+ExtraTrees only) validation accuracy:", soft_val_acc)

    # Pick whichever voting ensemble did best on validation, then finally
    # check it on the test set (only once — that's the whole point of
    # holding out a separate test set).
    best_voting = voting_clf_hard if hard_val_acc >= soft_val_acc else voting_clf_soft
    test_acc = best_voting.score(X_test, y_test)
    print("Ex8 - Best voting ensemble test accuracy:", test_acc)
    print("Ex8 - Individual classifiers' test accuracies:",
          {e.__class__.__name__: e.score(X_test, y_test) for e in estimators})
    # Typically the voting ensemble edges out every individual classifier by
    # a small margin (often <1%), since RF and ExtraTrees are already strong
    # and fairly correlated with each other on this task.
    return estimators, named_estimators, best_voting, test_acc



In [5]:
# ---------------------------------------------------------------------------
# 9a. Manual stacking: train a blender on validation-set predictions
# ---------------------------------------------------------------------------
def exercise_9_manual_blender(estimators, X_valid, y_valid, X_test, y_test,
                              random_state=42):
    # Run each individual classifier on the validation set, stack their
    # predictions column-wise into a new "meta-feature" training set.
    X_valid_predictions = np.empty((len(X_valid), len(estimators)), dtype=np.float32)
    for index, estimator in enumerate(estimators):
        X_valid_predictions[:, index] = estimator.predict(X_valid)

    blender = RandomForestClassifier(
        n_estimators=200, oob_score=True, random_state=random_state
    )
    blender.fit(X_valid_predictions, y_valid)
    print("Ex9 - Blender OOB score (on validation predictions):",
          blender.oob_score_)

    # Evaluate the full stack on the test set: run each classifier on the
    # test set, feed those predictions into the blender.
    X_test_predictions = np.empty((len(X_test), len(estimators)), dtype=np.float32)
    for index, estimator in enumerate(estimators):
        X_test_predictions[:, index] = estimator.predict(X_test)

    y_pred = blender.predict(X_test_predictions)
    stacking_test_acc = accuracy_score(y_test, y_pred)
    print("Ex9 - Manual stacking ensemble test accuracy:", stacking_test_acc)
    # In the book's run this comes out close to (sometimes very slightly
    # below) the voting classifier's test accuracy — stacking isn't
    # guaranteed to beat voting, especially when the base learners are
    # already fairly correlated (as RF and ExtraTrees are here).
    return blender, stacking_test_acc



In [8]:
# ---------------------------------------------------------------------------
# 9b. Same idea, using sklearn's built-in StackingClassifier
# ---------------------------------------------------------------------------
def exercise_9_sklearn_stacking(named_estimators, X_train, y_train,
                                X_test, y_test, random_state=42):
    # StackingClassifier handles the "train base learners with cross-val,
    # collect out-of-fold predictions, train a final estimator on those"
    # workflow internally — no need to manually split off a validation set
    # the way the from-scratch version does.
    stacking_clf = StackingClassifier(
        estimators=named_estimators,
        final_estimator=LogisticRegression(max_iter=1000, random_state=random_state),
        cv=3,  # keep small for speed on MNIST-sized data
        n_jobs=-1,
    )
    stacking_clf.fit(X_train, y_train)
    sklearn_stacking_test_acc = stacking_clf.score(X_test, y_test)
    print("Ex9 - sklearn StackingClassifier test accuracy:",
          sklearn_stacking_test_acc)
    return stacking_clf, sklearn_stacking_test_acc


if __name__ == "__main__":
    print("Loading MNIST (this requires network access to openml.org)...")
    X_train, y_train, X_valid, y_valid, X_test, y_test = load_mnist_splits()

    print("\n=== Exercise 8 ===")
    estimators, named_estimators, best_voting, voting_test_acc = exercise_8(
        X_train, y_train, X_valid, y_valid, X_test, y_test
    )

    print("\n=== Exercise 9 (manual blender) ===")
    blender, manual_stacking_acc = exercise_9_manual_blender(
        estimators, X_valid, y_valid, X_test, y_test
    )

    print("\n=== Exercise 9 (sklearn StackingClassifier) ===")
    stacking_clf, sklearn_stacking_acc = exercise_9_sklearn_stacking(
        named_estimators, X_train, y_train, X_test, y_test
    )

    print("\n=== Summary ===")
    print("Voting ensemble test acc:      ", voting_test_acc)
    print("Manual stacking test acc:      ", manual_stacking_acc)
    print("sklearn StackingClassifier acc:", sklearn_stacking_acc)


Loading MNIST (this requires network access to openml.org)...

=== Exercise 8 ===
Training RandomForestClassifier
Training ExtraTreesClassifier
Training LinearSVC
Ex8 - Individual validation accuracies: {'RandomForestClassifier': 0.9692, 'ExtraTreesClassifier': 0.9709, 'LinearSVC': 0.9116}
Ex8 - Hard voting validation accuracy: 0.9687
Ex8 - Soft voting (RF+ExtraTrees only) validation accuracy: 0.972
Ex8 - Best voting ensemble test accuracy: 0.9702
Ex8 - Individual classifiers' test accuracies: {'RandomForestClassifier': 0.965, 'ExtraTreesClassifier': 0.97, 'LinearSVC': 0.9122}

=== Exercise 9 (manual blender) ===
Ex9 - Blender OOB score (on validation predictions): 0.9695
Ex9 - Manual stacking ensemble test accuracy: 0.9652

=== Exercise 9 (sklearn StackingClassifier) ===


/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.che

Ex9 - sklearn StackingClassifier test accuracy: 0.9727

=== Summary ===
Voting ensemble test acc:       0.9702
Manual stacking test acc:       0.9652
sklearn StackingClassifier acc: 0.9727
